<a href="https://colab.research.google.com/github/karye/Liu-labbar/blob/main/TIC-TAC-TOE_lektioner/Lab_2_Lektion_6_Minimax.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎮 Tre i rad – Lektion 6: Minimax – Perfekt AI 🤖

**Målgrupp:** Gymnasiet, inga förkunskaper i programmering krävs  
**Tid:** ca 60 minuter  
**Mål:** Förstå Minimax-algoritmen, se hur perfekt spel fungerar och varför Minimax aldrig kan förlora

---

**Upphovspersoner:** LiU / Tekniksprånget  
**Licens:** [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/deed.sv)

---

### 📚 Förutsättning
Den här lektionen bygger vidare på **Lektion 5** (DFS & BFS).

### 🗺️ Vad gör vi idag?
1. Förstår **problemet** med DFS/BFS 🤔
2. Lär oss idén bakom **Minimax** 🧠
3. Bygger **poängsystemet** med `evaluate_board()` 📊
4. Implementerar `minimax()` med detaljerade kommentarer 💻
5. **Spelar** mot den perfekta AI:n och försöker vinna 🎮
6. **Jämför** alla agenter mot varandra 🏆
7. Löser övningar, tar quiz och summerar **hela kursen** 📝


In [ ]:
# ============================================================
# INSTÄLLNINGAR – Kör denna cell FÖRST!
# Innehåller all spellogik från alla tidigare lektioner.
# ============================================================

import ipywidgets as widgets
from IPython.display import display, clear_output
import random
import copy
import time
from collections import deque

BOARD_SIZE = 3   # Spelplanen är alltid 3x3

# ----------------------------------------------------------
# check_winner – kollar om någon spelare vann
# Returnerar: 1, 2, 3 (oavgjort), eller 0 (pågår)
# ----------------------------------------------------------
def check_winner(board):
    """Returnerar 1, 2, 3 (oavgjort) eller 0 (spelet pågår)."""
    for rad in range(BOARD_SIZE):
        if board[rad][0] != 0 and board[rad][0] == board[rad][1] == board[rad][2]:
            return board[rad][0]
    for kol in range(BOARD_SIZE):
        if board[0][kol] != 0 and board[0][kol] == board[1][kol] == board[2][kol]:
            return board[0][kol]
    if board[0][0] != 0 and board[0][0] == board[1][1] == board[2][2]:
        return board[0][0]
    if board[0][2] != 0 and board[0][2] == board[1][1] == board[2][0]:
        return board[0][2]
    if all(board[i][j] != 0 for i in range(BOARD_SIZE) for j in range(BOARD_SIZE)):
        return 3
    return 0

def make_move(board, row, col, player):
    """Returnerar True om draget lyckades, False annars."""
    if board[row][col] == 0:
        board[row][col] = player
        return True
    return False

def visa_braede(board):
    """Skriver ut spelplanen med symboler i terminalen."""
    symboler = {0: ".", 1: "X", 2: "O"}
    print("  0 1 2")
    for rad in range(BOARD_SIZE):
        rad_text = f"{rad} "
        for kol in range(BOARD_SIZE):
            rad_text += symboler[board[rad][kol]] + " "
        print(rad_text)
    print()

def find_empty_cells(board):
    """Returnerar lista med tomma (rad, kol) positioner."""
    tomma = []
    for rad in range(BOARD_SIZE):
        for kol in range(BOARD_SIZE):
            if board[rad][kol] == 0:
                tomma.append((rad, kol))
    return tomma

def random_agent(board, player=None):
    """Väljer ett slumpmässigt drag bland de tomma cellerna."""
    tomma_celler = find_empty_cells(board)
    if tomma_celler:
        return random.choice(tomma_celler)
    return None

def rule_based_agent(board, player):
    """Regelbaserad agent: Vinn, Blockera, Ta mitten, Ta hörn, Ta slump."""
    motstandare = 2 if player == 1 else 1
    for (rad, kol) in find_empty_cells(board):
        board[rad][kol] = player
        if check_winner(board) == player:
            board[rad][kol] = 0
            return (rad, kol)
        board[rad][kol] = 0
    for (rad, kol) in find_empty_cells(board):
        board[rad][kol] = motstandare
        if check_winner(board) == motstandare:
            board[rad][kol] = 0
            return (rad, kol)
        board[rad][kol] = 0
    if board[1][1] == 0:
        return (1, 1)
    for (rad, kol) in [(0,0), (0,2), (2,0), (2,2)]:
        if board[rad][kol] == 0:
            return (rad, kol)
    return random_agent(board)

def dfs_agent(board, player):
    """DFS-agent: Djupet-forst sokning fran Lektion 5."""
    def dfs(braede, spelare):
        vinnare = check_winner(braede)
        if vinnare == player:
            return 1, None
        elif vinnare != 0 and vinnare != player:
            return -1, None
        elif vinnare == 3:
            return 0, None
        basta_p = -2
        basta_d = None
        for (r, k) in find_empty_cells(braede):
            braede[r][k] = spelare
            nasta = 2 if spelare == 1 else 1
            p, _ = dfs(braede, nasta)
            p = -p
            braede[r][k] = 0
            if p > basta_p:
                basta_p = p
                basta_d = (r, k)
        return basta_p, basta_d
    _, drag = dfs(board, player)
    return drag

def play_game(agent1, agent2, verbose=False):
    """Spelar ett komplett spel. Returnerar vinnaren: 1, 2 eller 3."""
    braede = [[0]*BOARD_SIZE for _ in range(BOARD_SIZE)]
    agenter = [agent1, agent2]
    spelare = [1, 2]
    for drag_nr in range(BOARD_SIZE * BOARD_SIZE):
        agent = agenter[drag_nr % 2]
        spel = spelare[drag_nr % 2]
        drag = agent(braede, spel)
        if drag is None:
            break
        make_move(braede, drag[0], drag[1], spel)
        if verbose:
            visa_braede(braede)
        resultat = check_winner(braede)
        if resultat != 0:
            return resultat
    return 3

def run_tournament(agent1, agent2, antal_spel=100):
    """Kör ett turneringsläge. Returnerar (v1, v2, oavgjorda)."""
    v1 = v2 = o = 0
    for _ in range(antal_spel):
        r = play_game(agent1, agent2)
        if r == 1:
            v1 += 1
        elif r == 2:
            v2 += 1
        else:
            o += 1
    return v1, v2, o

print("✅ Alla funktioner fran tidigare lektioner ar laddade!")
print("🤖 Redo att bygga Minimax – den perfekta AI:n!")


## 🤔 Del 1: Problemet med DFS och BFS

### Vad kan DFS/BFS?

I Lektion 5 lärde vi oss DFS och BFS. De är kraftfulla:
- ✅ Kan söka igenom spelträdet
- ✅ Hittar vinnande drag

### Men vad kan de INTE?

DFS/BFS hittar drag som *kan* leda till vinst, men de räknar med att  
motståndaren spelar **slumpmässigt** eller gör **misstag** längs vägen.

```
Exempel – DFS väljer ett drag som bara vinner om O gör ett misstag:

  0 1 2
0 X . O      DFS kanske väljer (1,0) och hoppas att O missar (0,2)...
1 . . .      ...men O är smart och tar (0,2) och vinner!
2 . . .
```

### 🧠 Vad Minimax gör annorlunda

> **Minimax tänker som BÅDA spelarna:**  
> "Vad är det BÄSTA jag kan göra, *med antagandet att  
> motståndaren OCKSÅ spelar optimalt*?"

Det är skillnaden mellan att spela mot en dålig spelare och mot en stormästare!


## 🧠 Del 2: Idén bakom Minimax

### MAX och MIN

I Minimax finns det **två typer av noder** i spelträdet:

```
Vår tur (MAX) – Vi väljer HÖGST poäng
        |
   _____|_____
   |     |    |
  D.A   D.B  D.C
  p=3   p=5  p=-2
        ↑
   Vi väljer Drag B (p=5)! Det är MAX.

Motståndarens tur (MIN) – De väljer LÄGST poäng (sämst för oss!)
        |
   _____|_____
   |          |
  p=3        p=1
             ↑
   De väljer p=1 (sämre för oss). Det är MIN.
```

### Varför är detta realistiskt?

Vi antar att motståndaren spelar **optimalt** – de gör alltid det bästa möjliga.  
Om vi väljer ett drag som bara vinner om motståndaren gör ett misstag...  
...är det ett **dåligt drag** enligt Minimax!

### Poängsystemet

| Resultat | Poäng |
|----------|-------|
| Vi vinner ✅ | +10 |
| Oavgjort 🤝 | 0 |
| Vi förlorar ❌ | -10 |

*Djupet subtraheras/adderas – föredrar snabba vinster och sena förluster.*


## 📊 Del 3: Poängsystemet – evaluate_board()

Innan vi kan implementera Minimax behöver vi en funktion som  
**utvärderar hur bra ett spelläge är** för oss.

Den heter `evaluate_board()` och returnerar en numerisk poäng.


In [ ]:
# ============================================================
# EVALUATE_BOARD – Utvärderar ett spelläge med poäng
# ============================================================

def evaluate_board(board, player):
    """
    Utvärderar spelplanen och ger en poäng ur VÄRT perspektiv.

    Argument:
        board  – den spelplan vi utvärderar
        player – vi (1 eller 2)

    Returnerar:
        +10  om vi vinner
         0   om det är oavgjort eller spelet pågår
        -10  om vi förlorar
    """
    vinnare = check_winner(board)

    if vinnare == player:
        return 10        # ✅ Vi vinner – högt poäng!

    elif vinnare == 3:
        return 0         # 🤝 Oavgjort – neutralt

    elif vinnare != 0 and vinnare != player:
        return -10       # ❌ Motståndaren vinner – lågt poäng!

    return 0             # Spelet pågår – vet inte ännu

print("✅ evaluate_board() är redo!")
print()

# Testa poängsystemet på tre olika slutlägen
print("🧪 Test 1: X vinner (vi är X, spelare 1)")
b1 = [[1, 1, 1], [2, 2, 0], [0, 0, 0]]
visa_braede(b1)
print(f"  evaluate_board = {evaluate_board(b1, 1)}  (förväntat: +10)")

print()
print("🧪 Test 2: O vinner (vi är X, spelare 1)")
b2 = [[2, 2, 2], [1, 1, 0], [0, 0, 0]]
visa_braede(b2)
print(f"  evaluate_board = {evaluate_board(b2, 1)}  (förväntat: -10)")

print()
print("🧪 Test 3: Oavgjort")
b3 = [[1, 2, 1], [2, 1, 2], [2, 1, 2]]
visa_braede(b3)
print(f"  evaluate_board = {evaluate_board(b3, 1)}  (förväntat: 0)")


## 💻 Del 4: minimax() – Hjärtat i algoritmen

Nu implementerar vi Minimax! Läs kommentarerna noggrant.

### Pseudokod:

```
minimax(bräde, djup, maximerar, spelare):

    # Basfallet – spelet är klart
    om vi vinner: returnera +10 - djup
    om vi förlorar: returnera -10 + djup
    om oavgjort: returnera 0

    om maximerar (VÅR TUR – vi väljer HÖGST):
        bästa = -oändlighet
        för varje drag:
            gör draget
            poäng = minimax(bräde, djup+1, FALSKT, spelare)
            ångra draget
            bästa = max(bästa, poäng)
        returnera bästa

    annars (MOTSTÅNDARENS TUR – de väljer LÄGST):
        bästa = +oändlighet
        för varje drag:
            gör draget
            poäng = minimax(bräde, djup+1, SANT, spelare)
            ångra draget
            bästa = min(bästa, poäng)
        returnera bästa
```


In [ ]:
# ============================================================
# MINIMAX-ALGORITMEN – Perfekt AI för Tre i rad
# ============================================================

def minimax(board, djup, maximerar, player):
    """
    Minimax-algoritmen: Soker igenom hela speltradet och
    returnerar den OPTIMALA poangen, under antagandet att
    BADA spelarna spelar perfekt.

    Argument:
        board      – aktuell spelplan (andras och aterstalls)
        djup       – hur djupt vi ar i tradet (borjar pa 0)
        maximerar  – True = var tur (MAX), False = motstandardens tur (MIN)
        player     – vi (1 eller 2)

    Returnerar:
        Poang for det basta draget fran detta spelläge.
        Positiv = bra for oss, Negativ = daligt for oss.
    """

    # Identifiera motståndaren
    motstandare = 2 if player == 1 else 1

    # === BASFALLET: Kolla om spelet är klart ===
    poang = evaluate_board(board, player)

    if poang == 10:
        # Vi vinner! Subtrahera djupet – föredrar SNABBA vinster.
        # Djup 0 → 10, Djup 1 → 9, Djup 2 → 8, osv.
        return poang - djup

    if poang == -10:
        # Vi förlorar! Addera djupet – föredrar SENA förluster.
        # Djup 0 → -10, Djup 1 → -9, Djup 2 → -8, osv.
        return poang + djup

    # Oavgjort: Brädet är fullt men ingen vann
    if all(board[i][j] != 0 for i in range(BOARD_SIZE) for j in range(BOARD_SIZE)):
        return 0

    # === REKURSIVT FALL ===

    if maximerar:
        # ─────────────────────────────────────────────────────
        # DET ÄR VÅR TUR (MAX-spelare)
        # Vi väljer draget med HÖGST poäng
        # ─────────────────────────────────────────────────────
        basta = -100    # Börja med sämsta möjliga värde

        for (rad, kol) in find_empty_cells(board):
            # Steg 1: Gör draget för OSS
            board[rad][kol] = player

            # Steg 2: Rekursivt anrop
            #         Nu är det MOTSTÅNDARENS tur (maximerar=False)
            poang = minimax(board, djup + 1, False, player)

            # Steg 3: Ångra draget (backtracking)
            board[rad][kol] = 0

            # Steg 4: Uppdatera bästa om detta drag var bättre
            basta = max(basta, poang)

        return basta

    else:
        # ─────────────────────────────────────────────────────
        # DET ÄR MOTSTÅNDARENS TUR (MIN-spelare)
        # De väljer draget med LÄGST poäng (sämst för oss!)
        # ─────────────────────────────────────────────────────
        basta = 100     # Börja med bästa möjliga värde

        for (rad, kol) in find_empty_cells(board):
            # Steg 1: Gör draget för MOTSTÅNDAREN
            board[rad][kol] = motstandare

            # Steg 2: Rekursivt anrop
            #         Nu är det VÅR tur igen (maximerar=True)
            poang = minimax(board, djup + 1, True, player)

            # Steg 3: Ångra draget (backtracking)
            board[rad][kol] = 0

            # Steg 4: Uppdatera – motståndaren minimerar poängen
            basta = min(basta, poang)

        return basta


def minimax_agent(board, player):
    """
    Minimax-agenten: Väljer det BÄSTA draget med Minimax.
    Kan ALDRIG förlora i Tre i rad!
    """
    basta_poang = -100
    basta_drag = None

    # Prova alla möjliga drag och välj det med högst Minimax-poäng
    for (rad, kol) in find_empty_cells(board):
        # Gör draget
        board[rad][kol] = player

        # Kör Minimax: djup=0, maximerar=False (motståndarens tur)
        poang = minimax(board, 0, False, player)

        # Ångra draget
        board[rad][kol] = 0

        # Spara det bästa draget
        if poang > basta_poang:
            basta_poang = poang
            basta_drag = (rad, kol)

    return basta_drag

print("✅ minimax() och minimax_agent() är redo!")
print()

print("🧪 Test 1 – X kan vinna direkt:")
t1 = [[1, 1, 0], [2, 0, 0], [0, 0, 2]]
visa_braede(t1)
print(f"  Minimax väljer: {minimax_agent(t1, 1)}  (förväntat: (0,2))")

print()
print("🧪 Test 2 – O hotar vinna, X MÅSTE blockera:")
t2 = [[2, 2, 0], [1, 0, 0], [0, 0, 0]]
visa_braede(t2)
print(f"  Minimax väljer: {minimax_agent(t2, 1)}  (förväntat: (0,2))")

print()
print("🧪 Test 3 – Gaffeldrag (O i mitten):")
t3 = [[0, 0, 0], [0, 2, 0], [0, 0, 0]]
visa_braede(t3)
drag = minimax_agent(t3, 1)
print(f"  Minimax väljer: {drag}")
print("  (Bör ta ett hörn för att undvika gaffeldrag)")


## 🔬 Del 5: Stega igenom Minimax

Låt oss följa Minimax steg för steg på ett spelläge nära slutet!

**Spelplan (X:s tur):**
```
  0 1 2
0 X O X
1 O X O
2 . . X
```
Bara **2 tomma celler** kvar: (2,0) och (2,1).  
Kan X vinna, eller slutar det oavgjort?


In [ ]:
# ============================================================
# STEGA IGENOM MINIMAX – Visualisering
# ============================================================

sp_braede = [
    [1, 2, 1],
    [2, 1, 2],
    [0, 0, 1]
]

print("Startsituation:")
visa_braede(sp_braede)
print(f"Tomma celler: {find_empty_cells(sp_braede)}")
print()
print("=" * 52)
print("MINIMAX SOKER (X = spelare 1, O = spelare 2):")
print("=" * 52)

for (rad, kol) in find_empty_cells(sp_braede):
    print(f"\n[Niva 0 - Var tur MAX] Provar X pa ({rad},{kol}):")
    sp_braede[rad][kol] = 1

    vinnare = check_winner(sp_braede)
    if vinnare == 1:
        p = 10 - 0
        print(f"  -> X vinner direkt! Poang = 10 - 0 = +{p}")
        visa_braede(sp_braede)
    elif vinnare == 3:
        print(f"  -> Oavgjort! Poang = 0")
        visa_braede(sp_braede)
    else:
        visa_braede(sp_braede)
        print(f"  -> Spelet fortsatter, O:s tur (MIN):")
        for (r2, k2) in find_empty_cells(sp_braede):
            print(f"    [Niva 1 - Motstandardens tur MIN] Provar O pa ({r2},{k2}):")
            sp_braede[r2][k2] = 2
            v2 = check_winner(sp_braede)
            if v2 == 2:
                p2 = -10 + 1
                print(f"      -> O vinner! Poang = -10 + 1 = {p2}")
            elif v2 == 3:
                print(f"      -> Oavgjort! Poang = 0")
            else:
                print(f"      -> Spelet pagar, poang = 0")
            visa_braede(sp_braede)
            sp_braede[r2][k2] = 0

    sp_braede[rad][kol] = 0

print("=" * 52)
sp_braede2 = [[1,2,1],[2,1,2],[0,0,1]]
basta = minimax_agent(sp_braede2, 1)
print(f"\nMinimax valjer: {basta}")
print("Forklaring: Minimax valjer det drag som maximerar var poang!")


## 🎮 Del 6: Kan DU vinna mot Minimax?

Nu är det dags att testa din skicklighet mot den perfekta AI:n!

> 💡 **Minimax spelar perfekt – den kan aldrig förlora.**  
> Det bästa du kan uppnå är **oavgjort** om du spelar perfekt.  
> Om du gör ett enda misstag... förlorar du! 😄

Du spelar som **O** (spelare 2), Minimax spelar som **X** (spelare 1).


In [ ]:
# ============================================================
# INTERAKTIVT SPEL MOT MINIMAX
# ============================================================

def spela_mot_minimax():
    """Skapar ett interaktivt spel mot Minimax-agenten med knappar."""

    braede = [[0]*BOARD_SIZE for _ in range(BOARD_SIZE)]
    spel_slut = [False]
    symboler = {0: "⬜", 1: "❌", 2: "⭕"}
    knappar = [[None]*BOARD_SIZE for _ in range(BOARD_SIZE)]
    forsok = [0]
    vinster = [0]

    status_label = widgets.Label(
        value="🎮 Kan du slå Minimax? Du är ⭕  |  Klicka för att börja!"
    )
    poang_label = widgets.Label(value="📊 Försök: 0  |  Vinster: 0")

    def uppdatera_knappar():
        for r in range(BOARD_SIZE):
            for k in range(BOARD_SIZE):
                knappar[r][k].description = symboler[braede[r][k]]
                knappar[r][k].disabled = (braede[r][k] != 0) or spel_slut[0]

    def minimax_tur():
        drag = minimax_agent(braede, 1)
        if drag:
            make_move(braede, drag[0], drag[1], 1)
            uppdatera_knappar()
            vinnare = check_winner(braede)
            if vinnare == 1:
                status_label.value = "❌ Minimax vann! (Förväntat 😄) Klicka Försök igen!"
                spel_slut[0] = True
                for r in range(BOARD_SIZE):
                    for k in range(BOARD_SIZE):
                        knappar[r][k].disabled = True
            elif vinnare == 3:
                status_label.value = "🤝 Oavgjort! Du spelade perfekt! 🌟"
                spel_slut[0] = True
                for r in range(BOARD_SIZE):
                    for k in range(BOARD_SIZE):
                        knappar[r][k].disabled = True

    def klick(b, rad, kol):
        if spel_slut[0] or braede[rad][kol] != 0:
            return
        make_move(braede, rad, kol, 2)
        uppdatera_knappar()
        vinnare = check_winner(braede)
        if vinnare == 2:
            vinster[0] += 1
            status_label.value = "⭕ DU VANN! 🎉🏆 Det ska vara omöjligt... Kolla koden! 😅"
            spel_slut[0] = True
            poang_label.value = f"📊 Försök: {forsok[0]}  |  Vinster: {vinster[0]}"
            return
        elif vinnare == 3:
            status_label.value = "🤝 Oavgjort! Perfekt spelat! 🌟"
            spel_slut[0] = True
            for r in range(BOARD_SIZE):
                for k in range(BOARD_SIZE):
                    knappar[r][k].disabled = True
            return
        status_label.value = "🤖 Minimax tänker... (millisekunder!)"
        minimax_tur()
        if not spel_slut[0]:
            status_label.value = "🎮 Din tur! Du är ⭕  |  Tänk noga!"

    grid_rader = []
    for r in range(BOARD_SIZE):
        rad_knappar = []
        for k in range(BOARD_SIZE):
            knapp = widgets.Button(
                description="⬜",
                layout=widgets.Layout(width="65px", height="65px"),
            )
            knapp.on_click(lambda b, rad=r, kol=k: klick(b, rad, kol))
            knappar[r][k] = knapp
            rad_knappar.append(knapp)
        grid_rader.append(widgets.HBox(rad_knappar))

    nytt_spel = widgets.Button(
        description="🔄 Försök igen",
        button_style="warning",
        layout=widgets.Layout(width="140px", margin="8px 0px")
    )

    def aterstall(b):
        forsok[0] += 1
        for r in range(BOARD_SIZE):
            for k in range(BOARD_SIZE):
                braede[r][k] = 0
        spel_slut[0] = False
        uppdatera_knappar()
        status_label.value = "🎮 Nytt försök! Du är ⭕  |  Klicka för att börja!"
        poang_label.value = f"📊 Försök: {forsok[0]}  |  Vinster: {vinster[0]}"

    nytt_spel.on_click(aterstall)

    display(widgets.VBox([
        widgets.Label(value="─── 🤖 Minimax – Perfekt AI ───"),
        status_label,
        poang_label,
        *grid_rader,
        nytt_spel,
        widgets.Label(value="💡 Spoiler: Bästa möjliga resultat för dig är OAVGJORT!")
    ]))

spela_mot_minimax()


## 📊 Del 7: Stor jämförelse – Alla agenter!

Nu jämför vi alla agenter vi har byggt under kursen.


In [ ]:
# ============================================================
# STOR JÄMFÖRELSE – ALLA AGENTER MOT SLUMPMÄSSIG
# ============================================================

print("🏆 STOR AGENTTURNERING")
print("=" * 62)
print("Kör 100 spel per agent vs slumpmässig agent...")
print("(Minimax-spelen tar lite längre tid)")
print()

agenter = [
    ("Slumpmässig  🎲", random_agent),
    ("Regelbaserad 📋", rule_based_agent),
    ("DFS          🔍", dfs_agent),
    ("Minimax      🤖", minimax_agent),
]

print(f"{'Agent':22s} | {'Vinst':6s} | {'Förlust':7s} | {'Oavgjort':8s} | {'Tid':8s}")
print("-" * 62)

for namn, agent in agenter:
    start = time.time()
    v, f, o = run_tournament(agent, random_agent, 100)
    tid = time.time() - start
    print(f"{namn:22s} | {v:6d} | {f:7d} | {o:8d} | {tid:.2f}s")

print("=" * 62)
print()

# Direktmöten
print("⚔️  Direktmöten (50 spel per matchup):")
print()
start = time.time()
v1, v2, o = run_tournament(minimax_agent, dfs_agent, 50)
tid = time.time() - start
print(f"Minimax (sp.1) vs DFS (sp.2):")
print(f"  Minimax: {v1}  |  DFS: {v2}  |  Oavgjort: {o}  |  {tid:.1f}s")

print()
start = time.time()
v1, v2, o = run_tournament(minimax_agent, rule_based_agent, 50)
tid = time.time() - start
print(f"Minimax (sp.1) vs Regelbaserad (sp.2):")
print(f"  Minimax: {v1}  |  Regelbaserad: {v2}  |  Oavgjort: {o}  |  {tid:.1f}s")

print()
print("💡 Slutsatser:")
print("  Minimax kan aldrig förlora!")
print("  Men den är långsammare – söker hela spelträdet varje gång.")


## ✏️ Del 8: Övningar

Arbeta igenom dessa övningar för att befästa din förståelse av Minimax!

### 🟢 Grundläggande övningar


### Övning 1: Vad betyder maximerar=True?

**Fråga:** Förklara med egna ord vad `maximerar=True` respektive `maximerar=False` betyder.  
Kör koden nedan och verifiera med ett konkret exempel!


In [ ]:
# Övning 1 – Förstå parametern 'maximerar'

print("Demonstration av MAX och MIN i Minimax:")
print()

braede_test = [
    [1, 2, 1],
    [2, 1, 2],
    [0, 0, 0]   # 3 tomma celler
]

print("Spelplan:")
visa_braede(braede_test)

# Kör med maximerar=True (vår tur)
p_max = minimax(braede_test, 0, True, 1)
print(f"minimax(..., maximerar=True,  player=1) = {p_max}")
print(f"  -> Det är VÅR tur (MAX). Vi väljer det bästa draget för oss.")
print()

# Kör med maximerar=False (motståndarens tur)
p_min = minimax(braede_test, 0, False, 1)
print(f"minimax(..., maximerar=False, player=1) = {p_min}")
print(f"  -> Det är MOTSTÅNDARENS tur (MIN). De väljer sämsta för oss.")
print()
print("Sammanfattning:")
print("  maximerar=True  -> VÅR tur, vi maximerar poangen")
print("  maximerar=False -> MOTSTÅNDARENS tur, de minimerar poangen")


### Övning 2: Djupjustering i Minimax

`minimax()` justerar poängen med djupet:

```python
if poang == 10:
    return poang - djup   # Snabba vinster är bättre
if poang == -10:
    return poang + djup   # Sena förluster är bättre
```

**Fråga:** Vad är nyttan med detta? Kör koden och förklara!


In [ ]:
# Övning 2 – Djupjustering

print("Poängskala med djupjustering:")
print()
print("VINNANDE drag (poang - djup):")
for d in range(6):
    print(f"  Vinn pa djup {d}: poang = 10 - {d} = {10-d}")
print()
print("FORLORANDE drag (poang + djup):")
for d in range(6):
    print(f"  Forlust pa djup {d}: poang = -10 + {d} = {-10+d}")
print()
print("Slutsats:")
print("  Snabba vinster (litet djup) ger högt poäng – Minimax väljer dem!")
print("  Sena förluster (stort djup) ger minst straff – Minimax drar ut!")
print()

# Demonstrera
braede_demo = [[0, 1, 1], [0, 2, 0], [0, 0, 0]]
print("Demo – X kan vinna på (0,0) direkt:")
visa_braede(braede_demo)
drag = minimax_agent(braede_demo, 1)
print(f"Minimax väljer: {drag}")
print("(Minimax väljer (0,0) – den SNABBASTE vinsten!)")


### 🟡 Medelsvår övning

### Övning 3: Räkna Minimax-anrop

Hur många gånger anropas `minimax()` för ett tomt bräde?  
Lägg till en räknare och ta reda på det!


In [ ]:
# Övning 3 – Räkna Minimax-anrop

raknare = [0]   # Lista för att ändra inuti nested function

def minimax_raknare(board, djup, maximerar, player):
    """Minimax med inbyggd raknare."""
    raknare[0] += 1   # Räkna varje anrop

    motstandare = 2 if player == 1 else 1
    poang = evaluate_board(board, player)
    if poang == 10:
        return poang - djup
    if poang == -10:
        return poang + djup
    if all(board[i][j] != 0 for i in range(BOARD_SIZE) for j in range(BOARD_SIZE)):
        return 0

    if maximerar:
        basta = -100
        for (r, k) in find_empty_cells(board):
            board[r][k] = player
            p = minimax_raknare(board, djup + 1, False, player)
            board[r][k] = 0
            basta = max(basta, p)
        return basta
    else:
        basta = 100
        for (r, k) in find_empty_cells(board):
            board[r][k] = motstandare
            p = minimax_raknare(board, djup + 1, True, player)
            board[r][k] = 0
            basta = min(basta, p)
        return basta

# Räkna för ett tomt bräde
tomt = [[0]*3 for _ in range(3)]
raknare[0] = 0
start = time.time()
for (r, k) in find_empty_cells(tomt):
    tomt[r][k] = 1
    minimax_raknare(tomt, 0, False, 1)
    tomt[r][k] = 0
tid = time.time() - start

import math
print(f"Antal minimax-anrop for tomt brade: {raknare[0]:,}")
print(f"Tid: {tid:.2f}s")
print()
print(f"Maximalt mojliga spellagen (9!): {math.factorial(9):,}")
print(f"Utforskade: {raknare[0]:,} ({raknare[0]/math.factorial(9)*100:.1f}%)")
print()
print("Minimax utforskar FARRE an 9! lagen eftersom spelet slutar")
print("nar nagon vinner – inte alla grenar behover utforskas!")


### Övning 4: Kan Minimax förlora?

Verifiera empiriskt att Minimax **aldrig** kan förlora!  
Kör koden nedan med 200 spel per test.


In [ ]:
# Övning 4 – Verifiera Minimax är obesegrad

print("🧪 Verifiera: Kan Minimax förlora? (200 spel per test)")
print("   (Vänta lite – Minimax-spelen tar tid!)")
print()

# Som spelare 1 (X, börjar)
v1, f1, o1 = run_tournament(minimax_agent, random_agent, 200)
print(f"Minimax (spelare 1, X) vs Slump:")
print(f"  Vinst: {v1}  Förlust: {f1}  Oavgjort: {o1}")
if f1 == 0:
    print("  ✅ Minimax förlorade INTE en enda gång!")
else:
    print(f"  ⚠️  Minimax förlorade {f1} gånger – kolla koden!")

print()

# Som spelare 2 (O, börjar sist)
v2, f2, o2 = run_tournament(random_agent, minimax_agent, 200)
print(f"Slump vs Minimax (spelare 2, O):")
print(f"  Vinst: {v2}  Förlust: {f2}  Oavgjort: {o2}")
if f2 == 0:
    print("  ✅ Minimax förlorade INTE en enda gång som spelare 2 heller!")
else:
    print(f"  ⚠️  Minimax förlorade {f2} gånger som spelare 2!")

print()
print("Matematisk forklaring:")
print("  Tre i rad är ett 'löst spel'.")
print("  Minimax garanterar perfekt spel → kan aldrig förlora!")


### 🔴 Avancerad övning

### Övning 5: Designa ett eget poängsystem

Vad händer om vi lägger till **delpoäng** för bra positioner  
(t.ex. mitten, hörn)? Försök förbättra `evaluate_board()` och se  
om agenten väljer annorlunda på ett tomt bräde!


In [ ]:
# Övning 5 – Eget poängsystem

def evaluate_forbattrad(board, player):
    """
    Forbattrat poangssystem med delpoang for bra positioner.
    """
    vinnare = check_winner(board)
    if vinnare == player:
        return 10
    elif vinnare == 3:
        return 0
    elif vinnare != 0 and vinnare != player:
        return -10

    # Delpoäng för positioner (när spelet pågår)
    bonus = 0
    # Bonus för mitten
    if board[1][1] == player:
        bonus += 2
    elif board[1][1] != 0:
        bonus -= 1
    # Bonus för hörn
    for (r, k) in [(0,0), (0,2), (2,0), (2,2)]:
        if board[r][k] == player:
            bonus += 1
        elif board[r][k] != 0:
            bonus -= 1
    return bonus

def minimax_forbattrad(board, djup, maximerar, player):
    """Minimax med forbattrat poangssystem."""
    motstandare = 2 if player == 1 else 1
    poang = evaluate_forbattrad(board, player)
    if poang == 10:
        return poang - djup
    if poang == -10:
        return poang + djup
    if all(board[i][j] != 0 for i in range(BOARD_SIZE) for j in range(BOARD_SIZE)):
        return 0
    if maximerar:
        basta = -100
        for (r, k) in find_empty_cells(board):
            board[r][k] = player
            p = minimax_forbattrad(board, djup + 1, False, player)
            board[r][k] = 0
            basta = max(basta, p)
        return basta
    else:
        basta = 100
        for (r, k) in find_empty_cells(board):
            board[r][k] = motstandare
            p = minimax_forbattrad(board, djup + 1, True, player)
            board[r][k] = 0
            basta = min(basta, p)
        return basta

def minimax_agent_forbattrad(board, player):
    basta_p = -100
    basta_d = None
    for (r, k) in find_empty_cells(board):
        board[r][k] = player
        p = minimax_forbattrad(board, 0, False, player)
        board[r][k] = 0
        if p > basta_p:
            basta_p = p
            basta_d = (r, k)
    return basta_d

# Jämför på tomt bräde
tomt = [[0]*3 for _ in range(3)]
print("Tomt bräde – vad väljer varje version?")
d1 = minimax_agent(tomt, 1)
d2 = minimax_agent_forbattrad(tomt, 1)
print(f"  Original Minimax:   {d1}")
print(f"  Förbättrad Minimax: {d2}")
print()
print("Turneringsresultat (100 spel mot slump):")
v1, _, _ = run_tournament(minimax_agent, random_agent, 100)
v2, _, _ = run_tournament(minimax_agent_forbattrad, random_agent, 100)
print(f"  Original:   {v1}/100 vinster")
print(f"  Förbättrad: {v2}/100 vinster")
print()
print("Båda borde vinna lika ofta – Minimax är redan perfekt!")
print("Den förbättrade versionen kanske väljer mer strategiska drag!")


## 🚀 Bonus: Alpha-Beta Pruning – Minimax på steroider!

### Problemet med Minimax

Minimax utforskar **varje möjligt drag** i spelträdet.  
För Tre i rad är det okej – men för schack med 10^120 möjliga lägen? 😱

### Lösningen: Alpha-Beta Pruning (beskärning)

```
MAX väljer (vi söker högt):
       |
  _____|_____
  |          |
Drag A      Drag B
poäng=5     |
            MIN väljer (de söker lågt):
               |
          _____|_____
          |          |
         3           ?
         ↑
         3 < 5, så MIN väljer aldrig Drag B utifrån MAX.
         → Vi kan HOPPA ÖVER resten av Drag B! ✂️
```

`alfa` = det bästa MAX (vi) hittat hittills  
`beta` = det bästa MIN (motståndaren) hittat hittills  
Om `beta <= alfa` – hoppa över resten!

> 🔜 **I nästa kurs:**  
> Alpha-Beta Pruning på djupet, Monte Carlo Tree Search, och  
> Neurala nätverk som spelar spel (AlphaGo, AlphaZero!)


In [ ]:
# ============================================================
# BONUS: Alpha-Beta Pruning – En snabbare Minimax
# ============================================================

def minimax_ab(board, djup, alfa, beta, maximerar, player):
    """
    Minimax med Alpha-Beta Pruning.
    alfa = basta poang MAX hittat (borjar pa -100)
    beta = basta poang MIN hittat (borjar pa +100)
    Beskurning: om beta <= alfa, hoppa over resten!
    """
    motstandare = 2 if player == 1 else 1
    poang = evaluate_board(board, player)
    if poang == 10:
        return poang - djup
    if poang == -10:
        return poang + djup
    if all(board[i][j] != 0 for i in range(BOARD_SIZE) for j in range(BOARD_SIZE)):
        return 0

    if maximerar:
        basta = -100
        for (r, k) in find_empty_cells(board):
            board[r][k] = player
            p = minimax_ab(board, djup + 1, alfa, beta, False, player)
            board[r][k] = 0
            basta = max(basta, p)
            alfa = max(alfa, basta)
            if beta <= alfa:
                break   # ✂️ Beta-beskurning!
        return basta
    else:
        basta = 100
        for (r, k) in find_empty_cells(board):
            board[r][k] = motstandare
            p = minimax_ab(board, djup + 1, alfa, beta, True, player)
            board[r][k] = 0
            basta = min(basta, p)
            beta = min(beta, basta)
            if beta <= alfa:
                break   # ✂️ Alfa-beskurning!
        return basta

def minimax_ab_agent(board, player):
    """Minimax-agent med Alpha-Beta Pruning."""
    basta_p = -100
    basta_d = None
    for (r, k) in find_empty_cells(board):
        board[r][k] = player
        p = minimax_ab(board, 0, -100, 100, False, player)
        board[r][k] = 0
        if p > basta_p:
            basta_p = p
            basta_d = (r, k)
    return basta_d

print("✅ minimax_ab_agent() (Alpha-Beta) är redo!")
print()
print("⚡ Hastighetsjämförelse: Minimax vs Alpha-Beta (50 spel vardera)")
print("=" * 58)

start = time.time()
v1, _, _ = run_tournament(minimax_agent, random_agent, 50)
tid_mm = time.time() - start
print(f"Minimax:      {v1}/50 vinster | {tid_mm:.2f}s | {tid_mm/50*1000:.0f}ms/spel")

start = time.time()
v2, _, _ = run_tournament(minimax_ab_agent, random_agent, 50)
tid_ab = time.time() - start
print(f"Alpha-Beta:   {v2}/50 vinster | {tid_ab:.2f}s | {tid_ab/50*1000:.0f}ms/spel")

print("=" * 58)
forbattring = tid_mm / max(tid_ab, 0.001)
print(f"Alpha-Beta är {forbattring:.1f}x snabbare med EXAKT samma resultat!")
print("Alpha-Beta Pruning är en gratis optimering av Minimax.")


## 📝 Quiz – Testa dina kunskaper om Minimax!

Kör varje cell och kontrollera dina svar.


In [ ]:
# ❓ Fråga 1: Vad är MAX-spelaren i Minimax?
# a) Motståndaren – de maximerar sina poäng mot oss
# b) Vi – vi maximerar vår poäng
# c) Det beror på vem som börjar spelet

mitt_svar = "b"   # <- Andrade till a, b eller c

if mitt_svar == "b":
    print("✅ Rätt! MAX = vi. Vi väljer alltid draget med HÖGST poäng.")
    print("   MIN = motståndaren. De väljer draget med LÄGST poäng (sämst för oss).")
else:
    print(f"❌ '{mitt_svar}' är fel. Rätt svar är 'b': Vi är MAX-spelaren.")


In [ ]:
# ❓ Fråga 2: Vad returnerar evaluate_board() om vi vinner?
# a) True
# b) 0
# c) +10

mitt_svar = "c"   # <- Andra till a, b eller c

if mitt_svar == "c":
    print("✅ Rätt! evaluate_board() returnerar +10 om vi vinner.")
    print("   Poängsystemet: +10 = Vinst, 0 = Oavgjort, -10 = Förlust")
else:
    print(f"❌ '{mitt_svar}' är fel. Rätt svar är 'c': +10")


In [ ]:
# ❓ Fråga 3: Varför subtraherar Minimax djupet från vinnande poäng?
# a) Det är ett slumpmässigt designval utan praktisk betydelse
# b) För att Minimax ska föredra SNABBA vinster framför sena
# c) Djupet adderas, inte subtraheras

mitt_svar = "b"   # <- Andra till a, b eller c

if mitt_svar == "b":
    print("✅ Rätt! Djupsubtraktion gör att Minimax föredrar snabba vinster.")
    print("   Vinn pa djup 0: 10-0=10  vs  Vinn pa djup 3: 10-3=7")
    print("   -> Minimax väljer alltid den SNABBASTE möjliga vinsten!")
else:
    print(f"❌ '{mitt_svar}' är fel. Rätt svar är 'b'.")


In [ ]:
# ❓ Fråga 4: Kan Minimax förlora i Tre i rad om den implementeras korrekt?
# a) Ja, om motståndaren spelar perfekt
# b) Ja, om spelplanen är 4x4 eller större
# c) Nej – Minimax spelar alltid perfekt och kan aldrig förlora

mitt_svar = "c"   # <- Andra till a, b eller c

if mitt_svar == "c":
    print("✅ Rätt! En korrekt Minimax kan ALDRIG förlora i Tre i rad.")
    print()
    print("  Tre i rad är ett 'löst spel':")
    print("  Med perfekt spel (Minimax) är resultatet alltid OAVGJORT")
    print("  om båda spelar perfekt, eller VINST för Minimax om motståndaren gör misstag.")
else:
    print(f"❌ '{mitt_svar}' är fel. Rätt svar är 'c'.")


In [ ]:
# ❓ Fråga 5: Vad är Alpha-Beta Pruning?
# a) En ny algoritm som ersätter Minimax helt
# b) En optimering av Minimax som hoppar över onödiga sökvägar
# c) En teknik för att lägga till slumpmässighet i Minimax

mitt_svar = "b"   # <- Andra till a, b eller c

if mitt_svar == "b":
    print("✅ Rätt! Alpha-Beta Pruning är en optimering av Minimax.")
    print("   Den ger EXAKT samma resultat men kan vara mycket snabbare.")
    print("   'Alfa' och 'Beta' håller koll pa de bästa hittills funna poängen.")
else:
    print(f"❌ '{mitt_svar}' är fel. Rätt svar är 'b'.")


In [ ]:
# ❓ Fråga 6: Vad är den viktigaste skillnaden mellan DFS och Minimax?
# a) DFS är en sökning, Minimax är inte en sökning
# b) DFS och Minimax är identiska algoritmer med olika namn
# c) DFS tar inte hänsyn till motståndarens optimala spel, Minimax gör det

mitt_svar = "c"   # <- Andra till a, b eller c

print("Jämförelse DFS vs Minimax:")
print()
print("DFS:")
print("  Letar efter drag som KAN leda till vinst.")
print("  Antar inte explicit att motstandaren spelar optimalt.")
print()
print("Minimax:")
print("  MAX-spelaren valjer HOGST poang (vi)")
print("  MIN-spelaren valjer LAGST poang (motstandaren)")
print("  Antar att BADA spelarna spelar perfekt.")
print()
if mitt_svar == "c":
    print("✅ Rätt! Minimax hanterar motståndarens optimala spel, DFS gör inte det fullt ut.")
else:
    print(f"❌ '{mitt_svar}' är fel. Rätt svar är 'c'.")


## 🎓 Sammanfattning – Hela kursen i Tre i rad! 🏆

### Du har nu slutfört alla 6 lektioner! Grattis! 🎉

---

### 📚 Kursöversikt – Från nybörjare till AI-expert

| Lektion | Titel | Vad vi lärde oss |
|---------|-------|-----------------|
| **1** | 🎮 Första spelet | Spelplan, loopar, print, funktioner |
| **2** | 👥 Spela själv | Inmatning, while-loopen, turordning |
| **3** | 🎲 Slumpmässig AI | Listor, random, `find_empty_cells()` |
| **4** | 📋 Regelbaserad AI | if-satser, prioritetsregler, vinna/blockera |
| **5** | 🔍 DFS & BFS | Spelträd, rekursion, kö, backtracking |
| **6** | 🤖 Minimax | MAX/MIN, evaluate, perfekt spel |

---

### 🤖 Resan från enkel till perfekt AI

```
Slumpmässig  →  Regelbaserad  →    DFS/BFS    →    Minimax
    🎲                📋             🔍🌊              🤖

 Vinner ~50%     Vinner ~90%    Vinner ~85-95%   KAN ALDRIG
  mot slump        mot slump      mot slump       FÖRLORA!
```

---

### 🧠 De tre stora idéerna

1. **Regelbaserad AI** är snabb men begränsad – kan inte planera framåt
2. **Sökning (DFS/BFS)** kan planera framåt men missar motståndarens optimala spel
3. **Minimax** löser problemet – antar alltid att motståndaren spelar perfekt

---

### 🔮 Vad händer härnäst i AI-världen?

| Spel/Problem | Teknik |
|-------------|--------|
| ♟️ Schack | Minimax + Alpha-Beta + Neural Networks |
| 🎲 Go | Monte Carlo Tree Search + Deep Learning |
| 🎮 Videospel | Reinforcement Learning |
| 🚗 Självkörande bilar | Sensordata + ML + Planeringsalgoritmer |
| 💊 Läkemedelsforskning | Protein-veckningsalgoritmer (AlphaFold!) |

---

### 🙏 Tack och lycka till!

*Du har lärt dig grunderna i hur AI fungerar – från enkla regler till optimala  
sökalgoritmerna. Det här är samma principer som används i riktiga AI-system!*

*LiU / Tekniksprånget – [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/deed.sv)*

---
**🚀 Nästa steg: Neurala nätverk och maskininlärning väntar dig!**
